# Magenta Marker CV EDA

Detect the magenta square marker on the TaskBoard with classical CV. This notebook is meant to decide whether HSV thresholding + contour geometry is stable enough for marker-based pose estimation.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import random
import re

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None


def find_ws_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "ais").exists() and (path / "data").exists():
            return path
        if (path / "pixi.toml").exists() and (path / "ais").exists():
            return path.parent
        nested = path / "ws_aic"
        if (nested / "src" / "ais").exists() and (nested / "data").exists():
            return nested
    raise RuntimeError("Could not find ws_aic root from current working directory.")


WS_ROOT = find_ws_root(Path.cwd().resolve())
DATASET_DIR = WS_ROOT / "data" / "magenta_marker_cv"
IMAGE_EXTS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}

print(f"WS_ROOT: {WS_ROOT}")
print(f"DATASET_DIR: {DATASET_DIR} exists={DATASET_DIR.exists()}")

## Parameters

Start with a broad magenta HSV range, then tighten it after checking mask overlays.

In [ ]:
# Dataset / sampling
SPLITS = ["train", "val"]
IMAGE_FILTER = ""  # e.g. "center", "left", "right", "ep00320"
N_SAMPLES = 24
SEED = 7
FIG_COLS = 4

# OpenCV HSV uses H in [0, 179]. Magenta is usually around 145-165.
HSV_LOWER = np.array([125, 45, 45], dtype=np.uint8)
HSV_UPPER = np.array([175, 255, 255], dtype=np.uint8)

# Mask cleanup and contour filtering
BLUR_KERNEL = 3
MORPH_KERNEL = 5
MORPH_CLOSE_ITERS = 2
MORPH_DILATE_ITERS = 1
MIN_AREA_PX = 80
MAX_AREA_FRAC = 0.08
MIN_RECT_AREA_PX = 120
MIN_EXTENT = 0.12  # contour area / minAreaRect area

# Whether to prefer a rectangular contour with 4 approx vertices over largest area.
PREFER_APPROX_QUAD = True

print("HSV_LOWER", HSV_LOWER.tolist())
print("HSV_UPPER", HSV_UPPER.tolist())

## Detection Helpers

In [ ]:
def camera_from_name(image_path: Path) -> str:
    match = re.search(r"_(left|center|right)$", image_path.stem)
    return match.group(1) if match else "unknown"


def episode_from_name(image_path: Path) -> str:
    match = re.search(r"(ep\d+)", image_path.stem)
    return match.group(1) if match else image_path.stem.rsplit("_", 1)[0]


def iter_images() -> list[tuple[str, Path]]:
    rows = []
    for split in SPLITS:
        image_dir = DATASET_DIR / "images" / split
        if not image_dir.exists():
            continue
        for path in sorted(image_dir.rglob("*")):
            if path.suffix.lower() not in IMAGE_EXTS:
                continue
            if IMAGE_FILTER and IMAGE_FILTER not in path.name:
                continue
            rows.append((split, path))
    return rows


def order_corners_tl_tr_br_bl(points: np.ndarray) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float64).reshape(4, 2)
    center = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    pts = pts[np.argsort(angles)]
    # sorted by angle gives cyclic order. Rotate so first is top-left-ish.
    scores = pts[:, 0] + pts[:, 1]
    start = int(np.argmin(scores))
    pts = np.roll(pts, -start, axis=0)
    # Ensure clockwise TL, TR, BR, BL in image coordinates.
    signed_area = 0.5 * np.sum(pts[:, 0] * np.roll(pts[:, 1], -1) - np.roll(pts[:, 0], -1) * pts[:, 1])
    if signed_area < 0:
        pts = pts[[0, 3, 2, 1]]
    return pts


def magenta_mask(image_bgr: np.ndarray) -> np.ndarray:
    image = image_bgr
    if BLUR_KERNEL and BLUR_KERNEL > 1:
        k = BLUR_KERNEL if BLUR_KERNEL % 2 == 1 else BLUR_KERNEL + 1
        image = cv2.GaussianBlur(image, (k, k), 0)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, HSV_LOWER, HSV_UPPER)
    if MORPH_KERNEL and MORPH_KERNEL > 1:
        kernel = np.ones((MORPH_KERNEL, MORPH_KERNEL), dtype=np.uint8)
        if MORPH_CLOSE_ITERS > 0:
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=MORPH_CLOSE_ITERS)
        if MORPH_DILATE_ITERS > 0:
            mask = cv2.dilate(mask, kernel, iterations=MORPH_DILATE_ITERS)
    return mask


def contour_record(contour: np.ndarray, image_shape: tuple[int, int, int]) -> dict | None:
    image_h, image_w = image_shape[:2]
    area = float(cv2.contourArea(contour))
    if area < MIN_AREA_PX or area > image_w * image_h * MAX_AREA_FRAC:
        return None

    rect = cv2.minAreaRect(contour)
    (cx, cy), (rw, rh), angle = rect
    rect_area = float(max(rw * rh, 1.0))
    if rect_area < MIN_RECT_AREA_PX:
        return None
    extent = area / rect_area
    if extent < MIN_EXTENT:
        return None

    peri = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.04 * peri, True).reshape(-1, 2)
    rect_corners = order_corners_tl_tr_br_bl(cv2.boxPoints(rect))
    approx_corners = None
    if len(approx) == 4:
        approx_corners = order_corners_tl_tr_br_bl(approx)

    return {
        "area_px": area,
        "rect_area_px": rect_area,
        "extent": extent,
        "center_x": float(cx),
        "center_y": float(cy),
        "rect_w": float(rw),
        "rect_h": float(rh),
        "angle_deg": float(angle),
        "approx_vertices": int(len(approx)),
        "corners_px": approx_corners if approx_corners is not None and PREFER_APPROX_QUAD else rect_corners,
        "rect_corners_px": rect_corners,
        "approx_corners_px": approx_corners,
    }


def detect_magenta_marker(image_bgr: np.ndarray) -> dict | None:
    mask = magenta_mask(image_bgr)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    for contour in contours:
        record = contour_record(contour, image_bgr.shape)
        if record is not None:
            candidates.append(record)
    if not candidates:
        return None
    # Prefer clean quads, then larger contour area.
    candidates.sort(
        key=lambda r: (r["approx_vertices"] == 4, r["area_px"]),
        reverse=True,
    )
    best = candidates[0]
    best["mask"] = mask
    best["candidate_count"] = len(candidates)
    return best

## Dataset Scan

In [ ]:
rows = []
failures = []
images = iter_images()

for split, image_path in images:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        failures.append({"split": split, "image": str(image_path), "reason": "read_failed"})
        continue
    det = detect_magenta_marker(image)
    if det is None:
        failures.append({"split": split, "image": str(image_path), "reason": "no_marker"})
        continue
    h, w = image.shape[:2]
    row = {
        "split": split,
        "camera": camera_from_name(image_path),
        "episode": episode_from_name(image_path),
        "image": str(image_path),
        "area_px": det["area_px"],
        "rect_area_px": det["rect_area_px"],
        "extent": det["extent"],
        "area_frac": det["area_px"] / float(w * h),
        "rect_w": det["rect_w"],
        "rect_h": det["rect_h"],
        "angle_deg": det["angle_deg"],
        "approx_vertices": det["approx_vertices"],
        "candidate_count": det["candidate_count"],
    }
    corners = det["corners_px"]
    for idx, (x, y) in enumerate(corners):
        row[f"corner{idx}_x"] = float(x)
        row[f"corner{idx}_y"] = float(y)
    rows.append(row)

print(f"images={len(images)} detections={len(rows)} failures={len(failures)}")
print("camera counts", dict(Counter(row["camera"] for row in rows)))
print("split counts", dict(Counter(row["split"] for row in rows)))
print("failure reasons", dict(Counter(row["reason"] for row in failures)))

if pd is not None:
    df = pd.DataFrame(rows)
    display(df.head())
    if len(df):
        display(df.groupby(["split", "camera"]).size().rename("detections"))
        display(df[["area_px", "area_frac", "rect_w", "rect_h", "extent", "approx_vertices", "candidate_count"]].describe())
else:
    df = rows
    for row in rows[:5]:
        print(row)

for failure in failures[:20]:
    print(failure)

## Mask and Overlay Samples

In [ ]:
def draw_detection(ax, image_bgr: np.ndarray, det: dict | None, title: str) -> None:
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(image_rgb)
    if det is not None:
        corners = det["corners_px"]
        ax.add_patch(Polygon(corners, closed=True, fill=False, linewidth=2.0, edgecolor="#ff00ff"))
        for idx, (x, y) in enumerate(corners):
            ax.scatter([x], [y], s=28, color="#00ffff")
            ax.text(x + 3, y - 3, str(idx), color="#00ffff", fontsize=9, weight="bold")
        ax.text(
            4,
            18,
            f"area={det['area_px']:.0f} extent={det['extent']:.2f} cand={det['candidate_count']}",
            color="white",
            fontsize=8,
            bbox={"facecolor": "black", "alpha": 0.5, "pad": 1, "edgecolor": "none"},
        )
    else:
        ax.text(4, 18, "NO DETECTION", color="red", fontsize=10, weight="bold")
    ax.set_title(title, fontsize=8)
    ax.axis("off")


sample_pool = images
rng = random.Random(SEED)
samples = rng.sample(sample_pool, k=min(N_SAMPLES, len(sample_pool)))
fig_rows = math.ceil(len(samples) / FIG_COLS)
fig, axes = plt.subplots(fig_rows, FIG_COLS, figsize=(FIG_COLS * 5, fig_rows * 4))
axes = np.array(axes).reshape(-1)

for ax, (split, image_path) in zip(axes, samples):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    det = detect_magenta_marker(image) if image is not None else None
    draw_detection(ax, image, det, f"{split}/{image_path.name}")

for ax in axes[len(samples):]:
    ax.axis("off")
plt.tight_layout()

In [ ]:
# Inspect masks for the same samples.
fig, axes = plt.subplots(fig_rows, FIG_COLS, figsize=(FIG_COLS * 5, fig_rows * 4))
axes = np.array(axes).reshape(-1)

for ax, (split, image_path) in zip(axes, samples):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask = magenta_mask(image) if image is not None else None
    ax.imshow(mask, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"mask {split}/{image_path.name}", fontsize=8)
    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")
plt.tight_layout()

## Corner Distribution

In [ ]:
if pd is None:
    raise ImportError("Install pandas or skip this cell.")
if df.empty:
    raise RuntimeError("No detections to plot.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df["area_px"].hist(ax=axes[0], bins=30)
axes[0].set_title("marker contour area px")
df["extent"].hist(ax=axes[1], bins=30)
axes[1].set_title("contour / rect extent")
df["candidate_count"].hist(ax=axes[2], bins=20)
axes[2].set_title("candidate contours per image")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(7, 6))
colors = ["#00ffff", "#ffcc00", "#00ff66", "#ff66ff"]
for idx, color in enumerate(colors):
    ax.scatter(df[f"corner{idx}_x"], df[f"corner{idx}_y"], s=10, alpha=0.65, color=color, label=f"corner{idx}")
ax.invert_yaxis()
ax.set_xlabel("pixel x")
ax.set_ylabel("pixel y")
ax.set_title("detected marker corner distribution")
ax.legend()
ax.grid(True, alpha=0.25)

## Save Debug Overlays

Run this only when you want files on disk.

In [ ]:
SAVE_DEBUG = False
DEBUG_DIR = Path("/tmp/magenta_marker_cv_debug")

if SAVE_DEBUG:
    DEBUG_DIR.mkdir(parents=True, exist_ok=True)
    for split, image_path in images:
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        det = detect_magenta_marker(image)
        overlay = image.copy()
        if det is not None:
            corners = det["corners_px"].astype(int)
            cv2.polylines(overlay, [corners], True, (255, 0, 255), 2)
            for idx, (x, y) in enumerate(corners):
                cv2.circle(overlay, (int(x), int(y)), 4, (255, 255, 0), -1)
                cv2.putText(overlay, str(idx), (int(x) + 4, int(y) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        out = DEBUG_DIR / split / image_path.name
        out.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out), overlay)
    print(f"wrote {DEBUG_DIR}")
else:
    print("Set SAVE_DEBUG=True to write overlays.")